In [6]:
import os
from urllib.parse import quote

import pandas as pd

# Onde estão as bases: funciona no Jupyter local e no Google Colab
if os.path.exists("dados/Homicídios Mulheres_capitais.csv"):
    PASTA_DADOS = "dados"                      # Jupyter aberto na pasta data-science/
elif os.path.exists("data-science/dados/Homicídios Mulheres_capitais.csv"):
    PASTA_DADOS = "data-science/dados"         # Jupyter aberto na raiz do repositório
else:
    # Colab: baixa direto do repositório no GitHub, sem precisar enviar arquivos
    PASTA_DADOS = "https://raw.githubusercontent.com/Trindadefelipe/PratoApp/main/data-science/dados"

# Onde salvar os arquivos gerados: na pasta dados/ local ou na pasta atual do Colab
PASTA_SAIDA = PASTA_DADOS if not PASTA_DADOS.startswith("http") else "."

def caminho(nome):
    # Nomes com acento e espaço precisam ser codificados quando vêm da internet
    return f"{PASTA_DADOS}/{quote(nome)}" if PASTA_DADOS.startswith("http") else f"{PASTA_DADOS}/{nome}"

print("Lendo as bases de:", PASTA_DADOS)

homicidios = pd.read_csv(caminho("Homicídios Mulheres_capitais.csv"), sep=",")
taxa = pd.read_csv(caminho("Taxa de Homicídios Mulheres_capitais.csv"), sep=",")

# Deixa os nomes das colunas simples
homicidios.columns = ["periodo", "cod_ibge", "homicidios"]
taxa.columns = ["periodo", "cod_ibge", "taxa"]

# "1989-01-15T00:00:00.000Z" vira só o ano: 1989
homicidios["ano"] = pd.to_datetime(homicidios["periodo"]).dt.year
taxa["ano"] = pd.to_datetime(taxa["periodo"]).dt.year

print(homicidios.shape, taxa.shape)
homicidios.head()

(918, 4) (918, 4)


,periodo,cod_ibge,homicidios,ano
0,1989-01-15T00:00:00.000Z,1100205,14,1989
1,1990-01-15T00:00:00.000Z,1100205,12,1990
2,1991-01-15T00:00:00.000Z,1100205,11,1991
3,1992-01-15T00:00:00.000Z,1100205,14,1992
4,1993-01-15T00:00:00.000Z,1100205,11,1993


In [7]:
capitais = {
    1100205: "Porto Velho", 1200401: "Rio Branco", 1302603: "Manaus",
    1400100: "Boa Vista", 1501402: "Belém", 1600303: "Macapá",
    1721000: "Palmas", 2111300: "São Luís", 2211001: "Teresina",
    2304400: "Fortaleza", 2408102: "Natal", 2507507: "João Pessoa",
    2611606: "Recife", 2704302: "Maceió", 2800308: "Aracaju",
    2927408: "Salvador", 3106200: "Belo Horizonte", 3205309: "Vitória",
    3304557: "Rio de Janeiro", 3550308: "São Paulo", 4106902: "Curitiba",
    4205407: "Florianópolis", 4314902: "Porto Alegre", 5002704: "Campo Grande",
    5103403: "Cuiabá", 5208707: "Goiânia", 5300108: "Brasília",
}

homicidios["capital"] = homicidios["cod_ibge"].map(capitais)
taxa["capital"] = taxa["cod_ibge"].map(capitais)

print("Códigos sem nome:", homicidios[homicidios["capital"].isna()]["cod_ibge"].unique())
homicidios.head()

Códigos sem nome: []


,periodo,cod_ibge,homicidios,ano,capital
0,1989-01-15T00:00:00.000Z,1100205,14,1989,Porto Velho
1,1990-01-15T00:00:00.000Z,1100205,12,1990,Porto Velho
2,1991-01-15T00:00:00.000Z,1100205,11,1991,Porto Velho
3,1992-01-15T00:00:00.000Z,1100205,14,1992,Porto Velho
4,1993-01-15T00:00:00.000Z,1100205,11,1993,Porto Velho


In [8]:
print(taxa.dtypes)
taxa.head()

periodo      object
cod_ibge      int64
taxa        float64
ano           int32
capital      object
dtype: object


,periodo,cod_ibge,taxa,ano,capital
0,1989-01-15T00:00:00.000Z,1100205,5.40,1989,Porto Velho
1,1990-01-15T00:00:00.000Z,1100205,4.40,1990,Porto Velho
2,1991-01-15T00:00:00.000Z,1100205,3.83,1991,Porto Velho
3,1992-01-15T00:00:00.000Z,1100205,5.07,1992,Porto Velho
4,1993-01-15T00:00:00.000Z,1100205,3.71,1993,Porto Velho


In [9]:
dados = homicidios.merge(
    taxa[["cod_ibge", "ano", "taxa"]],
    on=["cod_ibge", "ano"],
    how="left",
)
dados = dados[["ano", "cod_ibge", "capital", "homicidios", "taxa"]]

print(dados.shape)
print("Anos:", dados["ano"].min(), "a", dados["ano"].max())
print("Capitais:", dados["capital"].nunique())
print("Valores vazios por coluna:")
print(dados.isna().sum())
dados.head()

(918, 5)
Anos: 1989 a 2022
Capitais: 27
Valores vazios por coluna:
ano           0
cod_ibge      0
capital       0
homicidios    0
taxa          0
dtype: int64


,ano,cod_ibge,capital,homicidios,taxa
0,1989,1100205,Porto Velho,14,5.40
1,1990,1100205,Porto Velho,12,4.40
2,1991,1100205,Porto Velho,11,3.83
3,1992,1100205,Porto Velho,14,5.07
4,1993,1100205,Porto Velho,11,3.71


In [10]:
dados.to_csv(f"{PASTA_SAIDA}/capitais_homicidios_mulheres.csv", index=False)
print("Base unificada salva em:", f"{PASTA_SAIDA}/capitais_homicidios_mulheres.csv")

# Etapa 1: Governança, Ética e Aquisição dos Dados
**Responsável:** Alana Caled

## 1. Origem dos dados
- **Fonte primária:** Sistema de Informações sobre Mortalidade (SIM), do Ministério da Saúde (MS/SVSA)
- **Plataforma de acesso:** Atlas da Violência, do IPEA e do Fórum Brasileiro de Segurança Pública (FBSP)
- **Link:** https://www.ipea.gov.br/atlasviolencia/tema/61/
- **Tema:** Violência por Gênero
- **Séries utilizadas:** "Homicídios Mulheres" e "Taxa de Homicídios Mulheres"
- **Recorte:** 27 capitais brasileiras
- **Período:** 1989 a 2022 (dados anuais)
- **Formato:** CSV
- **Data do download:** 18/10/2026
- **Metadados:** disponíveis na página de cada série no Atlas da Violência (ícone de documento)

## 2. Dicionário de dados
Tabela final: `capitais_homicidios_mulheres.csv`

| Coluna | Descrição |
| :--- | :--- |
| ano | Ano de referência dos dados |
| cod_ibge | Código IBGE do município (7 dígitos) |
| capital | Nome da capital (incluído pela equipe a partir do código IBGE) |
| homicidios | Número de mulheres vítimas de homicídio no ano |
| taxa | Homicídios de mulheres para cada 100 mil mulheres |

## 3. Observações da coleta
- Os recortes por UF e por município apresentaram erro no site ("Nenhum dado disponível para download"). Por isso, a equipe utilizou o recorte por capitais.
- O site informa que os arquivos usam ponto e vírgula como separador, mas os arquivos baixados vieram separados por vírgula.
- A coluna de data vinha no formato "1989-01-15T00:00:00.000Z" e foi convertida apenas para o ano.
- As duas séries foram unidas em uma única tabela usando o código IBGE e o ano.

## 4. Qualidade inicial
- 918 registros: 27 capitais x 34 anos (1989 a 2022).
- Nenhum valor ausente nas colunas.
- Todos os códigos IBGE corresponderam a uma capital.

## 5. Licença de uso
Os dados são públicos e divulgados gratuitamente pelo IPEA e pelo FBSP na plataforma Atlas da Violência, a partir do Sistema de Informações sobre Mortalidade (SIM) do Ministério da Saúde. A equipe utiliza os dados apenas para fins acadêmicos, sempre citando as fontes.

## 6. LGPD e ética
- Os dados são agregados por capital e por ano.
- Não há nenhuma informação que identifique uma pessoa (nome, CPF, endereço ou idade individual).
- Por isso, o uso desta base é compatível com a Lei Geral de Proteção de Dados (Lei nº 13.709/2018).
- A equipe não coleta nem utiliza dados reais de usuárias do aplicativo. Os testes usam apenas personas fictícias.

## 7. Limitações
- **Registro da causa da morte:** algumas mortes violentas são registradas como "causa indeterminada" e não entram na contagem de homicídios. Por isso, o número real tende a ser maior.
- **Recorte geográfico:** os dados das capitais não representam o interior dos estados.
- **Indisponibilidade:** os recortes por UF e por município não estavam disponíveis para download na data da coleta.
- **Tipo de violência:** a base trata apenas de homicídios e não inclui agressões, ameaças ou violência psicológica.
- **Mudança de classificação:** de 1979 a 1995 os homicídios eram identificados pela CID-9 (códigos E960 a E978). A partir de 1996 passou a ser usada a CID-10. Por isso, comparações entre períodos antes e depois de 1996 devem ser feitas com cuidado.

## 8. Aviso
Qualquer indicador gerado a partir destes dados é uma estimativa acadêmica e **não substitui** atendimento policial, médico ou jurídico.

Em caso de emergência:
- **Ligue 180:** Central de Atendimento à Mulher (gratuito, 24h)
- **190:** Polícia Militar

# Etapa 2: Análise Exploratória dos Dados (EDA)
**Responsável:** Felipe Trindade

A análise exploratória investiga as características essenciais das duas variáveis numéricas do estudo:
- `homicidios`: contagem absoluta de mulheres vítimas de homicídio registradas em cada capital por ano.
- `taxa`: taxa de homicídios calculada por 100 mil habitantes do sexo feminino.

---

## 2.1 Medidas de Tendência Central e Dispersão
O cálculo das estatísticas descritivas básicas permite compreender a concentração dos dados e o nível de variabilidade das ocorrências entre as 27 capitais brasileiras ao longo dos 34 anos de registros (1989 a 2022).

In [ ]:
# Cálculo das Medidas Descritivas e Dispersão
vars_num = ["homicidios", "taxa"]

tabela_descritiva = pd.DataFrame({
    "Média": dados[vars_num].mean(),
    "Mediana": dados[vars_num].median(),
    "Moda": [dados["homicidios"].mode()[0], dados["taxa"].mode()[0]],
    "Desvio Padrão": dados[vars_num].std(),
    "Variância": dados[vars_num].var(),
    "Mínimo": dados[vars_num].min(),
    "Máximo": dados[vars_num].max(),
    "Amplitude": dados[vars_num].max() - dados[vars_num].min()
})

tabela_descritiva.round(2)

### Interpretação dos Resultados

1. **Tendência Central (Média, Mediana e Moda):**
   - **Homicídios (absoluto):** A média de homicídios é de **38,42**, enquanto a mediana é de **20,00** e a moda é **12,00**. O fato de a média ser substancialmente superior à mediana evidencia uma forte **assimetria positiva (à direita)**. Isso ocorre porque metrópoles de grande porte populacional (como São Paulo e Rio de Janeiro) registram centenas de casos por ano, puxando a média para cima, enquanto a maioria das capitais registra valores menores.
   - **Taxa por 100 mil habitantes:** A média da taxa é de **4,19** e a mediana é de **3,79**. A proximidade entre a média e a mediana demonstra que a taxa proporcional é uma métrica mais estável e representativa da realidade epidemiológica do que os números absolutos, controlando o efeito da escala populacional. A moda 0,00 reflete anos iniciais sem ocorrências registradas em capitais recém-criadas ou menores.

2. **Dispersão (Desvio Padrão, Variância e Amplitude):**
   - **Homicídios (absoluto):** O desvio padrão de **55,67** e a amplitude de **477,00** (variando de 0 a 477) refletem a heterogeneidade das capitais do país em tamanho de população. A variância amostral é expressiva (3.099,65).
   - **Taxa por 100 mil habitantes:** O desvio padrão é de **2,40** com amplitude de **18,59** (mínimo de 0,00 e máximo de 18,59). Essa amplitude acentuada na taxa indica que, mesmo controlando a população, existem capitais e anos específicos onde o risco de letalidade violenta contra a mulher foi mais de quatro vezes superior à média nacional.

Esses achados fundamentam a necessidade de focar as análises comparativas e os futuros modelos preditivos na **taxa proporcional**, e não apenas no volume absoluto.

---

## 2.2 Distribuição de Frequências (Tabela e Histogramas)
Construção das faixas de frequência por classes e visualização da distribuição de probabilidades para a `taxa` (por 100 mil) e `homicidios` absolutos.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual
sns.set_theme(style="whitegrid", font_scale=1.05)

# 1. Tabela de distribuição de frequências para a Taxa (classes de 3 em 3)
faixas_taxa = [0, 3, 6, 9, 12, 15, 20]
rotulos_taxa = ["0 a 3", "3 a 6", "6 a 9", "9 a 12", "12 a 15", "15 ou mais"]
dados["faixa_taxa"] = pd.cut(dados["taxa"], bins=faixas_taxa, labels=rotulos_taxa, include_lowest=True)

freq_taxa = pd.DataFrame({
    "Frequência Absoluta": dados["faixa_taxa"].value_counts().sort_index(),
    "Frequência Relativa (%)": (dados["faixa_taxa"].value_counts(normalize=True).sort_index() * 100).round(2)
})
display(freq_taxa)

# 2. Histogramas com curva de densidade (KDE)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(dados["taxa"], kde=True, color="#0F8A54", bins=15, ax=axes[0])
axes[0].set_title("Distribuição da Taxa de Homicídios (por 100 mil)", fontsize=13, weight="bold")
axes[0].set_xlabel("Taxa por 100 mil mulheres")
axes[0].set_ylabel("Frequência")

sns.histplot(dados["homicidios"], kde=True, color="#E2674D", bins=20, ax=axes[1])
axes[1].set_title("Distribuição do Número Absoluto de Homicídios", fontsize=13, weight="bold")
axes[1].set_xlabel("Homicídios (contagem anual)")
axes[1].set_ylabel("Frequência")

plt.tight_layout()
plt.show()

### Interpretação da Distribuição de Frequências
- **Concentração principal:** Mais de **78%** de todos os registros anuais das capitais situam-se em faixas de taxa entre **0 e 6 homicídios por 100 mil mulheres** (com o pico na faixa de 3 a 6).
- **Cauda longa à direita (assimetria):** O histograma da taxa revela uma cauda estendida para a direita, alcançando valores superiores a 15 por 100 mil em anos críticos. O histograma de homicídios absolutos é ainda mais concentrado na base (mais de 70% abaixo de 30 casos anuais), com raros casos extremos em capitais altamente populosas.

---

## 2.3 Evolução Temporal por Ano e Comparação entre Capitais
Análise da série histórica (1989 a 2022) e contraste entre as capitais brasileiras.

In [ ]:
# 1. Gráfico de Evolução Temporal (Média das Capitais ao longo dos anos)
evolucao_ano = dados.groupby("ano")["taxa"].mean().reset_index()

plt.figure(figsize=(12, 5))
plt.plot(evolucao_ano["ano"], evolucao_ano["taxa"], marker="o", color="#0F8A54", linewidth=2.2, label="Média das Capitais")
plt.axvline(x=2006, color="#E2674D", linestyle="--", linewidth=1.8, label="2006: Lei Maria da Penha")
plt.title("Evolução da Taxa Média de Homicídios de Mulheres (1989 - 2022)", fontsize=14, weight="bold")
plt.xlabel("Ano")
plt.ylabel("Taxa média por 100 mil mulheres")
plt.legend()
plt.tight_layout()
plt.show()

# 2. Comparação entre Capitais (Taxa Média Histórica)
ranking_capitais = dados.groupby("capital")["taxa"].mean().sort_values(ascending=True)

plt.figure(figsize=(10, 8))
cores_ranking = ["#FAC333" if v < 4 else "#E2674D" for v in ranking_capitais.values]
ranking_capitais.plot(kind="barh", color=cores_ranking)
plt.title("Média Histórica da Taxa de Homicídios por Capital (1989 - 2022)", fontsize=14, weight="bold")
plt.xlabel("Taxa média por 100 mil mulheres")
plt.ylabel("Capital")
plt.tight_layout()
plt.show()

### Interpretação da Evolução Histórica e Disparidade Regional
- **Série Histórica:** A taxa média apresentou oscilações graduais, com momentos de pico nos anos 1990 e 2000. A partir da sanção da **Lei Maria da Penha (2006)** e com a melhoria no aparelhamento das Delegacias da Mulher, observa-se uma estabilização com tendência recente de redução nas médias globais, embora com heterogeneidade regional.
- **Disparidade entre Capitais:** O gráfico comparativo comprova que a violência letal feminina possui um recorte geográfico marcante. Capitais como **Boa Vista, Maceió e Porto Velho** registraram médias históricas alarmantes (acima de 7 por 100 mil), enquanto **São Paulo e Florianópolis** apresentam taxas substancialmente inferiores.

---

## 2.4 Matriz de Correlação entre Variáveis
Verificação de associações lineares entre o ano de observação, o número absoluto de homicídios e a taxa proporcional.

In [ ]:
# Cálculo da Matriz de Correlação de Pearson
matriz_corr = dados[["ano", "homicidios", "taxa"]].corr(method="pearson")
display(matriz_corr.round(3))

# Heatmap de Correlação
plt.figure(figsize=(7, 5))
sns.heatmap(matriz_corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Matriz de Correlação Linear (Pearson)", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()

### Interpretação da Matriz de Correlação
1. **Correlação entre Homicídios e Taxa (r ≈ 0,47):** Há uma correlação positiva moderada. Embora o aumento absoluto de casos reflita em parte um aumento na taxa, capitais de pequeno e médio porte com poucas ocorrências podem apresentar taxas altíssimas devido ao denominador populacional menor.
2. **Correlação com o Ano:**
   - A correlação entre a `taxa` e o `ano` é baixa/próxima de zero (-0,03 a nível nacional agregado), indicando que a evolução da violência contra a mulher não segue uma linha puramente decrescente contínua no Brasil como um todo, variando em ciclos regionais específicos.
3. **Subsídio para a Etapa 3 (Tratamento) e Etapa 4 (Modelagem):**
   - Como os números absolutos sofrem influência direta da densidade demográfica, modelos estatísticos e preditivos de risco no aplicativo SAFE devem privilegiar a **taxa padronizada** e variáveis contextuais de apoio, prevenindo vieses que penalizem desproporcionalmente as grandes cidades.

# Etapa 3: Qualidade, Limpeza e Pré-processamento dos Dados
**Responsável:** Higor Bueno

Nesta etapa, realizamos o tratamento formal de qualidade, auditoria de integridade e engenharia de recursos sobre a base consolidada de 27 capitais brasileiras ao longo de 34 anos (1989 a 2022). O objetivo central é transformar os dados brutos analisados na EDA em um conjunto de dados refinado, padronizado e escalonado, ideal para alimentar os algoritmos de **Machine Learning** da Etapa 4 (Higor Domingos).

---

## 3.1 Auditoria de Integridade, Tipagem e Valores Ausentes
A robustez de qualquer modelo estatístico ou de aprendizado de máquina depende da consistência de seus registros. Realizamos a verificação formal de tipos primitivos, nulos e duplicatas.

In [ ]:
# 1. Verificação de integridade estrutural e valores ausentes
print("Dimensão do dataset:", dados.shape)
print("\n--- Contagem de Valores Ausentes (NaN/Null) ---")
print(dados.isna().sum())

print("\n--- Verificação de Registros Duplicados ---")
duplicados = dados.duplicated(subset=["ano", "cod_ibge"]).sum()
print(f"Total de registros duplicados (Ano + Código IBGE): {duplicados}")

# 2. Asseguração formal de tipagem de dados
dados["ano"] = dados["ano"].astype(int)
dados["cod_ibge"] = dados["cod_ibge"].astype(int)
dados["capital"] = dados["capital"].astype(str)
dados["homicidios"] = dados["homicidios"].astype(int)
dados["taxa"] = dados["taxa"].astype(float)

print("\n--- Tipagem confirmada das colunas ---")
print(dados.dtypes)

### Interpretação da Auditoria de Integridade
- **Ausência de Nulos:** Confirmou-se que não existem valores nulos ou faltantes em nenhuma das 5 colunas fundamentais (`0` valores nulos nos 918 registros).
- **Sem Duplicatas:** A chave composta `[ano, cod_ibge]` é única em todos os 918 registros (27 capitais × 34 anos).
- **Tipagem Estrita:** Tipos primitivos assegurados (inteiros para ano, código e contagem; ponto flutuante para a taxa padronizada e string para a capital).

---

## 3.2 Detecção e Análise de Outliers (Critério de Tukey / IQR)
A análise exploratória da Etapa 2 revelou picos históricos expressivos na taxa de homicídios (máximo de 23.47 por 100 mil). Aplicamos o método do **Intervalo Interquartil (IQR)** para identificar estatisticamente os registros considerados outliers e definir a estratégia de tratamento.

In [ ]:
# Cálculo do Intervalo Interquartil (IQR) para a variável "taxa"
Q1 = dados["taxa"].quantile(0.25)
Q3 = dados["taxa"].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = max(0.0, float(Q1 - 1.5 * IQR))
limite_superior = float(Q3 + 1.5 * IQR)

outliers_taxa = dados[(dados["taxa"] < limite_inferior) | (dados["taxa"] > limite_superior)]

print(f"Q1 (25%): {Q1:.2f} | Q3 (75%): {Q3:.2f} | IQR: {IQR:.2f}")
print(f"Limite Inferior: {limite_inferior:.2f} | Limite Superior: {limite_superior:.2f}")
print(f"Total de outliers detectados na taxa: {len(outliers_taxa)} registros ({len(outliers_taxa)/len(dados)*100:.2f}%)")

print("\nExemplo dos 5 maiores picos históricos (outliers):")
display(outliers_taxa.sort_values(by="taxa", ascending=False).head(5)[["ano", "capital", "homicidios", "taxa"]])

### Decisão Metodológica sobre o Tratamento de Outliers
> ⚠️ **Diretriz de Governança e Ética:**  
> Os valores extremos identificados (ex.: taxas superiores a 10 por 100 mil em capitais como Porto Velho, Boa Vista ou Maceió em anos críticos) **NÃO foram excluídos nem imputados arbitrariamente**. 
> 
> **Justificativa:** Tratam-se de fenômenos sociodemográficos e eventos de violência reais documentados pelo IPEA e Ministério da Saúde, e não de erros de coleta ou digitação. A remoção desses picos mascararia crises de segurança pública e desprotegeria exatamente os cenários de maior risco que o sistema SAFE se propõe a combater. A estratégia adotada para controlar a sensibilidade a esses extremos é o **escalonamento robusto e a categorização em faixas de risco** nas subetapas seguintes.

---

## 3.3 Engenharia de Recursos (Feature Engineering): Macrorregiões e Faixas de Risco
Para aumentar a capacidade preditiva e interpretativa dos modelos de Machine Learning (Etapa 4), derivamos duas novas variáveis determinantes:
1. `regiao`: Mapeamento da macrorregião geográfica oficial (Norte, Nordeste, Centro-Oeste, Sudeste, Sul) através do dígito inicial do código IBGE do município.
2. `faixa_risco`: Discretização da taxa em classes ordinais (*Baixo*, *Médio*, *Alto*, *Crítico*) utilizando os quartis históricos da distribuição nacional.

In [ ]:
# 1. Derivação da Macrorregião Geográfica a partir do primeiro dígito do Código IBGE
mapa_regioes = {
    1: "Norte",
    2: "Nordeste",
    3: "Sudeste",
    4: "Sul",
    5: "Centro-Oeste"
}

dados_proc = dados.copy()
dados_proc["regiao"] = dados_proc["cod_ibge"].astype(str).str[0].astype(int).map(mapa_regioes)

# 2. Discretização da Taxa em 4 Classes de Risco (Quartis Históricos)
rotulos_risco = ["Baixo", "Médio", "Alto", "Crítico"]
dados_proc["faixa_risco"] = pd.qcut(dados_proc["taxa"], q=4, labels=rotulos_risco)

print("Distribuição das Capitais por Macrorregião:")
print(dados_proc.groupby("regiao")["capital"].nunique())

print("\nDistribuição dos registros nas Faixas de Risco:")
print(dados_proc["faixa_risco"].value_counts(sort=False))
dados_proc.head()

---

## 3.4 Normalização e Escalonamento de Variáveis Numéricas
Algoritmos de aprendizado supervisionado e não-supervisionado (como K-Means, SVM, KNN e Redes Neurais) são altamente sensíveis a variáveis em escalas numéricas distintas. Enquanto a contagem absoluta de `homicidios` atinge centenas de casos em capitais populosas, a `taxa` situa-se geralmente abaixo de 10.
Aplicamos duas técnicas complementares do `scikit-learn`:
- **Min-Max Scaling (Normalização [0, 1]):** comprime os dados em um intervalo fixo entre 0 e 1, preservando as proporções relativas.
- **StandardScaler (Padronização Z-Score, $\mu=0, \sigma=1$):** centraliza na média zero e escala por variância unitária, ideal para algoritmos baseados em distâncias euclidianas.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Instanciação dos escalonadores
scaler_minmax = MinMaxScaler()
scaler_std = StandardScaler()

# Aplicação do Min-Max Scaling [0, 1]
dados_proc["taxa_normalizada"] = scaler_minmax.fit_transform(dados_proc[["taxa"]])
dados_proc["homicidios_normalizado"] = scaler_minmax.fit_transform(dados_proc[["homicidios"]])

# Aplicação da Padronização Z-Score
dados_proc["taxa_padronizada_z"] = scaler_std.fit_transform(dados_proc[["taxa"]])

# One-Hot Encoding da Macrorregião para uso em algoritmos de Machine Learning
dados_encoded = pd.get_dummies(dados_proc, columns=["regiao"], prefix="regiao", drop_first=False)

print("Estatísticas após Normalização Min-Max [0, 1]:")
display(dados_proc[["taxa_normalizada", "homicidios_normalizado"]].describe().round(3))
print("\nVisualização da base pré-processada com colunas normalizadas:")
dados_proc.head()

---

## 3.5 Exportação do Dataset Processado e Síntese Metodológica
O dataset final tratado foi estruturado e exportado para a pasta oficial de dados do projeto, ficando pronto para consumo imediato pela **Etapa 4 (Modelagem de Machine Learning - Higor Domingos)**.

In [ ]:
# Exportação do dataset limpo e enriquecido com as features prontas para ML
# PASTA_SAIDA vem da primeira célula: dados/ no Jupyter local, pasta atual no Colab
caminho_exportacao = f"{PASTA_SAIDA}/capitais_dados_processados.csv"
dados_proc.to_csv(caminho_exportacao, index=False)

print(f"Dataset processado exportado em {caminho_exportacao}: {dados_proc.shape[0]} linhas e {dados_proc.shape[1]} colunas.")
dados_proc.head(5)

### 🎯 Síntese da Etapa 3 e Entrega para a Etapa 4
1. **Integridade Validada:** Base de 918 registros sem nulos e com tipagem estrita (Issue #52).
2. **Outliers Contextualizados:** Picos históricos mantidos para garantir proteção nos cenários mais graves e não desproteger as vítimas (Issue #52).
3. **Novas Features Geradas:** Macrorregião geográfica e Faixa de Risco Ordinal adicionadas (Issue #53).
4. **Escalonamento Realizado:** Variáveis numéricas normalizadas em escala `[0, 1]` e Z-Score disponíveis para evitar viés de escala (Issue #53).
5. **Preparação para a Etapa 4 (Higor Domingos):** O arquivo gerado fornece tanto variáveis contínuas escalonadas (para regressão/clusterização K-Means) quanto rótulos discretos (`faixa_risco`) para algoritmos supervisionados de classificação.